In [6]:
from transformers import AutoImageProcessor, SwinForMaskedImageModeling
import torch
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

image_processor = AutoImageProcessor.from_pretrained("microsoft/swin-base-simmim-window6-192")
model = SwinForMaskedImageModeling.from_pretrained("microsoft/swin-base-simmim-window6-192")

num_patches = (model.config.image_size // model.config.patch_size) ** 2
pixel_values = image_processor(images=image, return_tensors="pt").pixel_values
# create random boolean mask of shape (batch_size, num_patches)
bool_masked_pos = torch.randint(low=0, high=2, size=(1, num_patches)).bool()

outputs = model(pixel_values, bool_masked_pos=bool_masked_pos)
loss, reconstructed_pixel_values = outputs.loss, outputs.reconstruction
list(reconstructed_pixel_values.shape)

preprocessor_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/360M [00:00<?, ?B/s]

[1, 3, 192, 192]

In [7]:
reconstructed_pixel_values.shape

torch.Size([1, 3, 192, 192])

In [10]:
import plotly.express as px

px.imshow(reconstructed_pixel_values[0].detach().cpu().numpy().swapaxes(0, 2))

In [11]:
bool_masked_pos = torch.randint(low=0, high=4, size=(1, num_patches)).__le__(4).bool()
with torch.no_grad():
    outputs = model(pixel_values, bool_masked_pos=bool_masked_pos)

In [13]:
import numpy as np
params = sum([np.prod(p.size()) for p in model.parameters()])
print("number of parameters:", str(params // 1e6) + "M")

number of parameters: 89.0M


In [12]:
px.imshow(reconstructed_pixel_values[0].detach().cpu().numpy().swapaxes(0, 2))